In [ ]:
# Исследование текстового датасета модерации (EDA + очистка + статистики)

from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Делаем удобные пути + reuse из пакета
import sys
sys.path.append(str(Path('..').resolve() / 'src'))

from pinz_ml.paths import data_dir

DATA_PATH = data_dir() / 'text_dataset.parquet'
assert DATA_PATH.exists(), f"Не найден parquet: {DATA_PATH}"

df = pd.read_parquet(DATA_PATH)
print('Rows:', len(df))
print('Columns:', list(df.columns))
print(df.dtypes)
display(df.head(10))

# 1) Общая сводка пропусков/дубликатов
missing = df.isna().mean().sort_values(ascending=False)
print('Missing ratio (top 20):')
display(missing.head(20))

if 'text' in df.columns and 'label' in df.columns:
    dup = df.duplicated(subset=['text', 'label']).mean()
    print('Duplicates(text,label) ratio:', float(dup))

# 2) Распределение классов
if 'label' in df.columns:
    vc = df['label'].value_counts().sort_index()
    plt.figure(figsize=(6, 3))
    vc.plot(kind='bar')
    plt.title('Label distribution')
    plt.xlabel('label')
    plt.ylabel('count')
    plt.tight_layout()
    plt.show()

# 3) Длины текста и выбросы
if 'text' in df.columns:
    s = df['text'].astype(str)
    df['len_chars'] = s.map(len)
    df['len_tokens'] = s.str.split().map(len)

    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    sns.histplot(df['len_chars'], bins=80, ax=ax[0])
    ax[0].set_title('len_chars')
    sns.histplot(df['len_tokens'], bins=80, ax=ax[1])
    ax[1].set_title('len_tokens')
    plt.tight_layout()
    plt.show()

    print('len_chars quantiles:')
    display(df['len_chars'].quantile([0.5, 0.9, 0.95, 0.99]))
    print('len_tokens quantiles:')
    display(df['len_tokens'].quantile([0.5, 0.9, 0.95, 0.99]))

# 4) Мини-очистка/нормализация и проверка примеров

def normalize_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"https?://\S+", " URL ", s)
    s = re.sub(r"@\w+", " USER ", s)
    s = re.sub(r"#\w+", " HASHTAG ", s)
    s = re.sub(r"\d+", " NUM ", s)
    s = re.sub(r"[^a-zа-я0-9\s]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

if 'text' in df.columns:
    df['text_norm'] = df['text'].map(normalize_text)
    display(df[['text', 'text_norm']].head(10))

# 5) Самые частые токены по классам (быстро, без больших моделей)
if 'label' in df.columns and 'text_norm' in df.columns:
    def top_tokens(sub: pd.Series, n=30):
        toks = ' '.join(sub.tolist()).split()
        c = pd.Series(toks).value_counts().head(n)
        return c

    labels = sorted(df['label'].unique().tolist())
    for lab in labels[:5]:  # ограничим вывод
        c = top_tokens(df[df['label'] == lab]['text_norm'], n=20)
        print(f"Top tokens for label={lab}")
        display(c)


In [ ]:
# Быстрый baseline + анализ ошибок на валидации

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

assert 'text_norm' in df.columns and 'label' in df.columns

X = df['text_norm'].values
y = df['label'].astype(int).values

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(splitter.split(X, y))

X_tr, y_tr = X[train_idx], y[train_idx]
X_va, y_va = X[val_idx], y[val_idx]

pipe = Pipeline(
    [
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=150_000, min_df=2)),
        ('clf', LogisticRegression(max_iter=300, n_jobs=1, class_weight='balanced')),
    ]
)

pipe.fit(X_tr, y_tr)
pred = pipe.predict(X_va)
print(classification_report(y_va, pred, digits=4))

# Примеры ошибок
proba = pipe.predict_proba(X_va)[:, -1] if hasattr(pipe, 'predict_proba') else None

err = pd.DataFrame({'text': X_va, 'y': y_va, 'pred': pred})
if proba is not None:
    err['score'] = proba

err_fp = err[(err['y'] == 0) & (err['pred'] == 1)].head(20)
err_fn = err[(err['y'] == 1) & (err['pred'] == 0)].head(20)

print('False positives (sample):')
display(err_fp)
print('False negatives (sample):')
display(err_fn)
